# Rayong Crop Classification — Random Forest 2-stage cascade

Sentinel-2 L2A → 15-class crop classification (14 target crops + OTHER) for Rayong Province, Thailand. Years 2018, 2020, 2024 (BE 2561 / 2563 / 2567).

Pipeline:
1. Discover Sentinel-2 `.SAFE.zip` per (year, month)
2. Build AOI grid (S2 tile T47PQQ ∩ shapefile bbox, 10 m)
3. Map LDD codes → 15 classes; drop urban / composite / mixed-plant / aquaculture / eucalyptus
4. Adaptive parcel buffer + 2-pass rasterise (target overrides OTHER)
5. Per-pixel features: 5 indices (NDVI / EVI / MNDWI / MTCI / SWIR) × 3 months + 10 deltas = 25
6. SCL cloud mask + ≥2-valid-month filter + cross-month NaN impute
7. Per-class cap + pixel-stratified 80/10/10 split
8. Stage-1 RF (15 cls) + Stage-2 tropical-orchard specialist (8 cls)
9. Pixel-level + parcel-vote evaluation

## 1. Config

In [ ]:
from __future__ import annotations
import gc, json, time, joblib, zipfile, warnings
from contextlib import ExitStack
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.windows import Window
import rasterio.windows
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

SEED       = 42
DATA_DIR   = Path('full_dataset_l2a')
LU_DIR     = Path('full_dataset') / 'Landuse_ryg'
S2_DIR     = DATA_DIR / 'rayong_raster'
OUT_PARQ   = Path('baseline_dataset.parquet')
OUT_META   = Path('baseline_dataset_meta.json')
ARTIFACTS  = Path('rf_artifacts'); ARTIFACTS.mkdir(exist_ok=True)
MODEL_PATH = ARTIFACTS / 'baseline_rf.joblib'

YEARS = {
    2018: LU_DIR / 'ระยอง61'    / 'LU_RYG_2561.shp',
    2020: LU_DIR / 'ระยอง63'    / 'LU_RYG_2563.shp',
    2024: LU_DIR / 'ระยอง2567' / 'การใช้ที่ดิน' / 'LU_RYG_2567.shp',
}

MONTHS     = {10: 'Oct', 11: 'Nov', 12: 'Dec'}
BAND_NAMES = ['B02', 'B03', 'B04', 'B05', 'B06', 'B08', 'B11']
SCL_BAND   = 'SCL'
S2_SCALE   = 1e-4

INDEX_NAMES  = ['ndvi', 'evi', 'mndwi', 'mtci', 'swir']
FEATURE_COLS = [f'{idx} {m}' for m in MONTHS for idx in INDEX_NAMES]

LABEL_NAMES = {
    2101: 'Paddy', 2204: 'Cassava', 2205: 'Pineapple', 2302: 'Rubber', 2303: 'OilPalm',
    2403: 'Durian', 2404: 'Rambutan', 2405: 'Coconut', 2407: 'Mango', 2413: 'Longan',
    2416: 'Jackfruit', 2419: 'Mangosteen', 2420: 'Langsat', 4201: 'Reservoir', 9999: 'OTHER',
}
TARGET_LABELS = [c for c in LABEL_NAMES if c != 9999]
OTHER_LABEL   = 9999
ALL_LABELS    = TARGET_LABELS + [OTHER_LABEL]

URBAN_RANGE = (1000, 2000)
LABEL_MERGE = {4101: 4201, 4103: 4201, 4202: 4201, 4102: 4201}

NON_TARGET_DROP_CODES = {
    2201, 2301, 2401, 2501, 2601, 2801, 2901, 2001,
    2100, 2200, 2300, 2400, 2500, 2700, 2900,
    2902, 2903, 2904, 2905,
    2304,
}

OTHER_POLICY = 'agriculture_only'

BUFFER_M               = -10
MIN_AREA_FOR_BUFFER_M2 = 400
MAX_PIXELS_PER_PARCEL  = 200
SCL_CLOUD_CLASSES      = {1, 8, 9, 10}
CHUNK_ROWS             = 1024
MAX_BAND_THREADS       = 6
MIN_VALID_MONTHS       = 2

RF_PARAMS = dict(
    n_estimators=300, max_depth=36, min_samples_split=5, min_samples_leaf=1,
    max_features='log2', class_weight='balanced_subsample', bootstrap=True,
    n_jobs=-1, random_state=SEED,
)

DEFAULT_CAP   = 400_000
PER_CLASS_CAP = {}

## 2. Sentinel-2 discovery

In [ ]:
def find_safe_zip(year, month_name):
    d = S2_DIR / str(year) / month_name
    return next(iter(sorted(d.glob('*.SAFE.zip'))), None) if d.exists() else None

def band_uri(zp, band):
    with zipfile.ZipFile(zp) as zf:
        names = [n for n in zf.namelist() if '/IMG_DATA/' in n and n.endswith('.jp2')]
        for res in ('10m', '20m', '60m'):
            for n in names:
                if f'_{band}_{res}' in n.split('/')[-1]:
                    return f'/vsizip/{zp.as_posix()}/{n}'
    return None

def discover_s2():
    out = {}
    for year, shp in YEARS.items():
        s2 = {}
        for m, name in MONTHS.items():
            zp = find_safe_zip(year, name)
            if zp is None:
                s2[m] = None; continue
            bands = {b: band_uri(zp, b) for b in BAND_NAMES + [SCL_BAND]}
            s2[m] = bands if all(bands.values()) else None
        out[year] = {'shp': shp, 's2': s2}
    return out

YEARS_CFG = discover_s2()
for year, cfg in YEARS_CFG.items():
    miss = [MONTHS[m] for m, v in cfg['s2'].items() if v is None]
    print(f'{year}: shp={cfg["shp"].exists()}  miss={miss or "none"}')

## 3. AOI grid

In [ ]:
def first_b02():
    for cfg in YEARS_CFG.values():
        for v in cfg['s2'].values():
            if v is not None:
                return v['B02']
    raise RuntimeError('no S2 band found')

with rasterio.open(first_b02()) as ref:
    REF_TF, REF_CRS = ref.transform, ref.crs
    TILE_H, TILE_W  = ref.height, ref.width
    TILE_BNDS       = ref.bounds

aoi = None
for cfg in YEARS_CFG.values():
    if not cfg['shp'].exists(): continue
    g = gpd.read_file(cfg['shp'], columns=['geometry'])
    if g.crs.to_string() != REF_CRS.to_string():
        g = g.to_crs(REF_CRS)
    b = g.total_bounds
    aoi = b.copy() if aoi is None else np.array([
        min(aoi[0], b[0]), min(aoi[1], b[1]),
        max(aoi[2], b[2]), max(aoi[3], b[3]),
    ])

ox, oy, res = REF_TF.c, REF_TF.f, REF_TF.a
col0 = max(int(np.floor((max(aoi[0], TILE_BNDS.left)   - ox) / res)), 0)
col1 = min(int(np.ceil ((min(aoi[2], TILE_BNDS.right)  - ox) / res)), TILE_W)
row0 = max(int(np.floor((oy - min(aoi[3], TILE_BNDS.top))    / res)), 0)
row1 = min(int(np.ceil ((oy - max(aoi[1], TILE_BNDS.bottom)) / res)), TILE_H)

AOI_WIN = Window(col_off=col0, row_off=row0, width=col1 - col0, height=row1 - row0)
AOI_TF  = rasterio.windows.transform(AOI_WIN, REF_TF)
H, W    = AOI_WIN.height, AOI_WIN.width
print(f'AOI: {H} x {W}  ({H*W/1e6:.1f} M px)  CRS={REF_CRS.to_string()}')

## 4. Vegetation indices

| index | formula |
|---|---|
| NDVI  | (B08 − B04) / (B08 + B04) |
| EVI   | 2.5 · (B08 − B04) / (B08 + 6·B04 − 7.5·B02 + 1) |
| MNDWI | (B03 − B11) / (B03 + B11) |
| MTCI  | (B06 − B05) / (B05 − B04) |
| SWIR  | B11 |

In [ ]:
def safe_div(a, b, eps=1e-10):
    bs = np.where(np.abs(b) > eps, b, 1.0)
    out = a / bs
    out[np.abs(b) <= eps] = 0.0
    return out

def compute_indices(B):
    nir, red, blu = B['B08'], B['B04'], B['B02']
    grn          = B['B03']
    re1, re2, sw = B['B05'], B['B06'], B['B11']
    nrd = nir - red
    return {
        'ndvi':  np.clip(safe_div(nrd, nir + red),                          -1.0, 1.0).astype(np.float32),
        'evi':   np.clip(2.5 * safe_div(nrd, nir + 6*red - 7.5*blu + 1.0),  -1.0, 1.0).astype(np.float32),
        'mndwi': np.clip(safe_div(grn - sw, grn + sw),                      -1.0, 1.0).astype(np.float32),
        'mtci':  np.clip(safe_div(re2 - re1, re1 - red),                    -5.0, 10.0).astype(np.float32),
        'swir':  sw.astype(np.float32),
    }

## 5. Label mapping + rasterise + per-year extraction

Drop logic: urban (LUL1='U' or 1xxx) | composite (LU_ID_L3>99999 or '/' in LU_CODE) | NON_TARGET_DROP_CODES (mixed/abandoned/aquaculture/eucalyptus). With `OTHER_POLICY='agriculture_only'` also drop F/M/non-target-W. Two-pass rasterise (target overrides OTHER); within each pass smaller parcels win.

In [ ]:
def map_labels_from_gdf(gdf):
    raw_codes = gdf['LU_ID_L3'].fillna(0).astype(np.int64).to_numpy()
    lul1      = gdf['LUL1_CODE'].fillna('').astype(str).to_numpy()
    has_slash = gdf['LU_CODE'].fillna('').str.contains('/', regex=False).to_numpy()

    primary = raw_codes.astype(np.int32)
    if LABEL_MERGE:
        max_code = int(max(primary.max(initial=0), max(LABEL_MERGE.keys()))) + 1
        lut = np.arange(max_code, dtype=np.int32)
        for s, dst in LABEL_MERGE.items():
            if s < max_code: lut[s] = dst
        primary = lut[np.clip(primary, 0, max_code - 1)]
    merged64 = primary.astype(np.int64)

    is_urban     = (lul1 == 'U') | ((raw_codes >= URBAN_RANGE[0]) & (raw_codes < URBAN_RANGE[1]))
    is_composite = (raw_codes > 99999) | has_slash
    is_drop_code = np.isin(raw_codes, list(NON_TARGET_DROP_CODES))
    drop = is_urban | is_composite | is_drop_code

    if OTHER_POLICY == 'agriculture_only':
        is_forest      = (lul1 == 'F')
        is_misc        = (lul1 == 'M')
        target_arr     = np.asarray(TARGET_LABELS, dtype=np.int64)
        is_water_other = (lul1 == 'W') & ~np.isin(merged64, target_arr)
        drop = drop | is_forest | is_misc | is_water_other

    out = np.where(np.isin(primary, TARGET_LABELS), primary, OTHER_LABEL).astype(np.int32)
    out[drop] = -1
    return out


def adaptive_buffer(gdf, buffer_m, min_area_m2):
    """Negative buffer for parcels >= min_area_m2; keep small parcels intact."""
    areas = gdf.geometry.area.to_numpy()
    big   = areas >= min_area_m2
    geoms = gdf.geometry.where(~big).combine_first(gdf.geometry.where(big).buffer(buffer_m))
    empty = geoms.is_empty | geoms.isna()
    geoms = geoms.where(~empty, gdf.geometry)
    gdf = gdf.copy()
    gdf.geometry = geoms
    return gdf


def rasterise_parcels(gdf):
    """Two-pass: OTHER background, then targets overwrite. Smaller-wins within pass."""
    is_target = (gdf['label'] != OTHER_LABEL).to_numpy()
    g_other   = gdf[~is_target].assign(_a=lambda d: d.geometry.area).sort_values('_a', ascending=False).reset_index(drop=True)
    g_target  = gdf[ is_target].assign(_a=lambda d: d.geometry.area).sort_values('_a', ascending=False).reset_index(drop=True)

    n_o, n_t = len(g_other), len(g_target)
    g_other  = g_other.assign(_p=np.arange(1,        n_o + 1,        dtype=np.int32))
    g_target = g_target.assign(_p=np.arange(n_o + 1, n_o + n_t + 1,  dtype=np.int32))

    pid_r = np.zeros((H, W), dtype=np.int32)
    if n_o:
        pid_r = rasterize(zip(g_other.geometry, g_other['_p']),
                          out_shape=(H, W), transform=AOI_TF, fill=0,
                          dtype='int32', all_touched=False)
    if n_t:
        pid_t = rasterize(zip(g_target.geometry, g_target['_p']),
                          out_shape=(H, W), transform=AOI_TF, fill=0,
                          dtype='int32', all_touched=False)
        mask_t = pid_t > 0
        pid_r = np.where(mask_t, pid_t, pid_r)

    pid_to_lab = np.zeros(n_o + n_t + 1, dtype=np.int32)
    if n_o: pid_to_lab[g_other['_p'].to_numpy()]  = g_other['label'].to_numpy(dtype=np.int32)
    if n_t: pid_to_lab[g_target['_p'].to_numpy()] = g_target['label'].to_numpy(dtype=np.int32)
    return pid_r, pid_to_lab[pid_r]


def per_parcel_cap(pid, max_per, seed):
    n = len(pid)
    rng = np.random.default_rng(seed)
    order = np.lexsort((rng.random(n), pid))
    starts = np.searchsorted(pid[order], pid[order], side='left')
    keep = np.empty(n, dtype=bool)
    keep[order] = (np.arange(n) - starts) < max_per
    return keep


STATS = {}

def _read_band(vrt, win, lr, lc):
    return vrt.read(1, window=win)[lr, lc]


def extract_year(year, cfg):
    if not cfg['shp'].exists():
        print(f'  {year}: SHP MISSING'); return None
    available = [m for m, v in cfg['s2'].items() if v is not None]
    if not available:
        print(f'  {year}: no months available'); return None

    gdf = gpd.read_file(cfg['shp'], columns=['LU_ID_L3', 'LU_CODE', 'LUL1_CODE', 'geometry'])
    if gdf.crs.to_string() != REF_CRS.to_string():
        gdf = gdf.to_crs(REF_CRS)

    raw       = gdf['LU_ID_L3'].fillna(0).astype(np.int64).to_numpy()
    lul1_arr  = gdf['LUL1_CODE'].fillna('').astype(str).to_numpy()
    lcode_arr = gdf['LU_CODE'].fillna('').astype(str)
    n_total   = len(gdf)
    n_urban   = int(((lul1_arr == 'U') | ((raw >= URBAN_RANGE[0]) & (raw < URBAN_RANGE[1]))).sum())
    n_compos  = int(((raw > 99999) | lcode_arr.str.contains('/', regex=False)).sum())
    n_mixed   = int(np.isin(raw, list(NON_TARGET_DROP_CODES)).sum())
    n_forest  = int((lul1_arr == 'F').sum())
    n_misc    = int((lul1_arr == 'M').sum())

    gdf['label'] = map_labels_from_gdf(gdf)
    n_dropped = int((gdf['label'] == -1).sum())
    gdf = gdf[gdf['label'] >= 0].reset_index(drop=True)
    print(f'  {year}: dropped {n_dropped:,}/{n_total:,} '
          f'(urban={n_urban:,} composite={n_compos:,} non-target={n_mixed:,} '
          f'forest={n_forest:,} misc={n_misc:,})')

    target_polys = {int(t): int((gdf['label'] == t).sum()) for t in TARGET_LABELS}

    n_pre = len(gdf)
    n_small = int((gdf.geometry.area < MIN_AREA_FOR_BUFFER_M2).sum())
    gdf = adaptive_buffer(gdf, BUFFER_M, MIN_AREA_FOR_BUFFER_M2)
    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notna()].reset_index(drop=True)
    print(f'  {year}: buffered {len(gdf):,}/{n_pre:,} parcels ({n_small:,} skipped buffer)')

    pid_r, lab_r = rasterise_parcels(gdf)
    valid = np.flatnonzero(pid_r.ravel() > 0)
    if len(valid) == 0:
        print(f'  {year}: no valid pixels'); return None

    rows = (valid // W).astype(np.int32)
    cols = (valid %  W).astype(np.int32)
    lab  = lab_r.ravel()[valid].astype(np.int32)
    pid  = pid_r.ravel()[valid].astype(np.int64) + year * 10_000_000

    keep = per_parcel_cap(pid, MAX_PIXELS_PER_PARCEL, SEED + year)
    rows, cols, lab, pid = rows[keep], cols[keep], lab[keep], pid[keep]
    order = np.argsort(rows, kind='stable')
    rows, cols, lab, pid = rows[order], cols[order], lab[order], pid[order]
    n = len(rows)
    print(f'  {year}: {n:,} pixels across {len(np.unique(pid)):,} parcels')

    X = np.full((n, len(FEATURE_COLS)), np.nan, dtype=np.float32)
    n_idx = len(INDEX_NAMES)
    n_cloud = {m: 0 for m in MONTHS}
    valid_month = np.zeros((n, len(MONTHS)), dtype=bool)

    with ExitStack() as stk:
        vrts, scls = {m: {} for m in available}, {}
        for m in available:
            for b in BAND_NAMES:
                src = stk.enter_context(rasterio.open(cfg['s2'][m][b]))
                vrts[m][b] = stk.enter_context(WarpedVRT(
                    src, crs=REF_CRS, transform=AOI_TF, height=H, width=W,
                    resampling=Resampling.bilinear))
            scl_src = stk.enter_context(rasterio.open(cfg['s2'][m][SCL_BAND]))
            scls[m] = stk.enter_context(WarpedVRT(
                scl_src, crs=REF_CRS, transform=AOI_TF, height=H, width=W,
                resampling=Resampling.nearest))
        pool = stk.enter_context(ThreadPoolExecutor(max_workers=MAX_BAND_THREADS))

        for r0 in tqdm(range(0, H, CHUNK_ROWS), desc=f'  {year}'):
            rh    = min(CHUNK_ROWS, H - r0)
            start = int(np.searchsorted(rows, r0,      side='left'))
            end   = int(np.searchsorted(rows, r0 + rh, side='left'))
            if start == end: continue
            lr  = (rows[start:end] - r0).astype(np.int32)
            lc  = cols[start:end]
            win = Window(col_off=0, row_off=r0, width=W, height=rh)

            for mi, m in enumerate(MONTHS):
                if m not in available: continue
                futs = [pool.submit(_read_band, vrts[m][b], win, lr, lc) for b in BAND_NAMES]
                raws = [f.result() for f in futs]
                scl_vals = scls[m].read(1, window=win)[lr, lc]

                nodata = np.zeros(end - start, dtype=bool)
                for raw in raws:
                    nodata |= (raw == 0)
                B = {b: raws[i].astype(np.float32) * S2_SCALE for i, b in enumerate(BAND_NAMES)}
                cloud_pix = np.isin(scl_vals, list(SCL_CLOUD_CLASSES))
                bad = nodata | cloud_pix
                n_cloud[m] += int((cloud_pix & ~nodata).sum())

                vi = compute_indices(B)
                block = np.column_stack([vi[name] for name in INDEX_NAMES])
                block[bad] = np.nan
                X[start:end, mi*n_idx:(mi+1)*n_idx] = block
                valid_month[start:end, mi] = ~bad
            del raws, B
            gc.collect()

    n_valid_per_pixel = valid_month.sum(axis=1)
    keep = n_valid_per_pixel >= MIN_VALID_MONTHS
    print(f'  {year}: kept {int(keep.sum()):,}/{n:,} (dropped {int((~keep).sum()):,} <{MIN_VALID_MONTHS}-valid)  cloud={n_cloud}')

    df = pd.DataFrame(X[keep], columns=FEATURE_COLS)
    df.insert(0, 'label', lab[keep])
    df['year']      = np.int16(year)
    df['row']       = rows[keep]
    df['col']       = cols[keep]
    df['parcel_id'] = pid[keep]
    df['n_valid_months'] = n_valid_per_pixel[keep].astype(np.int8)

    STATS[year] = dict(
        pixels_kept=int(keep.sum()),
        cloud_per_month=n_cloud,
        parcels_per_class=target_polys,
        urban_dropped=n_urban, composite_dropped=n_compos, non_target_dropped=n_mixed,
        forest_dropped=n_forest if OTHER_POLICY == 'agriculture_only' else 0,
        misc_dropped=n_misc if OTHER_POLICY == 'agriculture_only' else 0,
        small_parcels_unbuffered=n_small,
    )
    return df

## 6. Run all years + save parquet

In [ ]:
frames = []
for year, cfg in YEARS_CFG.items():
    print(f'\n=== {year} ===')
    df = extract_year(year, cfg)
    if df is not None and len(df):
        frames.append(df)
    gc.collect()
df_all = pd.concat(frames, ignore_index=True)
del frames; gc.collect()

# Cross-month NaN impute (n_valid_months == 2 only; <2 already dropped, 3 has no NaN)
n_filled = 0
for idx_name in INDEX_NAMES:
    cols = [f'{idx_name} {m}' for m in MONTHS]
    arr  = df_all[cols].to_numpy(dtype=np.float32, copy=True)
    for i in range(len(cols)):
        others = [j for j in range(len(cols)) if j != i]
        n_valid = (~np.isnan(arr[:, others])).sum(axis=1)
        sum_v   = np.nansum(arr[:, others], axis=1)
        mean_o  = np.where(n_valid > 0, sum_v / np.maximum(n_valid, 1), np.nan)
        target  = np.isnan(arr[:, i]) & ~np.isnan(mean_o)
        n_filled += int(target.sum())
        arr[target, i] = mean_o[target]
    df_all[cols] = arr
df_all = df_all.dropna(subset=FEATURE_COLS).reset_index(drop=True)

df_all[FEATURE_COLS] = df_all[FEATURE_COLS].astype(np.float32).round(3)
df_all['label']     = df_all['label'].astype(np.int32)
df_all['year']      = df_all['year'].astype(np.int16)
df_all['row']       = df_all['row'].astype(np.int32)
df_all['col']       = df_all['col'].astype(np.int32)
df_all['parcel_id'] = df_all['parcel_id'].astype(np.int64)
df_all['n_valid_months'] = df_all['n_valid_months'].astype(np.int8)
df_all = df_all[['label'] + FEATURE_COLS + ['year', 'row', 'col', 'parcel_id', 'n_valid_months']]
df_all.to_parquet(OUT_PARQ, compression='zstd', index=False)

OUT_META.write_text(json.dumps({
    'classes': ALL_LABELS, 'feature_cols': FEATURE_COLS, 'years': list(YEARS.keys()),
    'seed': SEED, 'buffer_m': BUFFER_M, 'min_area_for_buffer_m2': MIN_AREA_FOR_BUFFER_M2,
    'min_valid_months': MIN_VALID_MONTHS, 'urban_range': list(URBAN_RANGE),
    'scl_cloud_classes': sorted(SCL_CLOUD_CLASSES),
    'max_pixels_per_parcel': MAX_PIXELS_PER_PARCEL, 'cross_month_nan_fills': n_filled,
    'aoi': {'crs': REF_CRS.to_string(), 'transform': list(AOI_TF)[:6],
            'height': int(H), 'width': int(W)},
    'per_year_stats': STATS,
}, indent=2, ensure_ascii=False))

print(f'\nsaved -> {OUT_PARQ}  ({len(df_all):,} rows, {OUT_PARQ.stat().st_size/1e6:.1f} MB)')
print(f'cross-month NaN fills: {n_filled:,}')

## 7. Feature engineering + per-class cap + 80/10/10 split

In [ ]:
from sklearn.model_selection import train_test_split

df = pd.read_parquet(OUT_PARQ)

DELTA_NAMES = []
for idx in INDEX_NAMES:
    df[f'{idx} d11_10'] = (df[f'{idx} 11'] - df[f'{idx} 10']).astype(np.float32)
    df[f'{idx} d12_11'] = (df[f'{idx} 12'] - df[f'{idx} 11']).astype(np.float32)
    DELTA_NAMES += [f'{idx} d11_10', f'{idx} d12_11']
ALL_FEATS = FEATURE_COLS + DELTA_NAMES
print(f'features: {len(ALL_FEATS)} (15 base + 10 delta)')

y = df['label'].to_numpy()
rng = np.random.default_rng(SEED)
parts = []
for c in np.unique(y):
    cap = PER_CLASS_CAP.get(int(c), DEFAULT_CAP)
    idx = np.flatnonzero(y == c)
    parts.append(rng.choice(idx, cap, replace=False) if len(idx) > cap else idx)
keep = np.concatenate(parts); rng.shuffle(keep)
df = df.iloc[keep].reset_index(drop=True)
print(f'capped: {len(df):,} rows')

X   = df[ALL_FEATS].to_numpy(dtype=np.float32)
y   = df['label'].to_numpy(dtype=np.int32)
pid = df['parcel_id'].to_numpy(dtype=np.int64)
del df; gc.collect()

X_tv, X_te, y_tv, y_te, pid_tv, pid_te = train_test_split(
    X, y, pid, test_size=0.10, stratify=y, random_state=SEED)
X_tr, X_va, y_tr, y_va, pid_tr, pid_va = train_test_split(
    X_tv, y_tv, pid_tv, test_size=1/9, stratify=y_tv, random_state=SEED)
del X, y, pid, X_tv, y_tv, pid_tv; gc.collect()
print(f'train: {X_tr.shape}  val: {X_va.shape}  test: {X_te.shape}')

## 8. Stage-1 RF (15 classes)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

t0 = time.time()
clf = RandomForestClassifier(**RF_PARAMS).fit(X_tr, y_tr)
print(f'stage-1 fit: {time.time()-t0:.1f}s  classes={clf.classes_.tolist()}')
CLF_CLASSES = np.asarray(clf.classes_, dtype=np.int32)

## 9. Stage-2 specialist (8 tropical-orchard classes)

In [ ]:
TROPICAL_ORCHARDS = np.array([2403, 2404, 2405, 2407, 2413, 2416, 2419, 2420], dtype=np.int32)

mask_tr_orch = np.isin(y_tr, TROPICAL_ORCHARDS)
X_tr_orch, y_tr_orch = X_tr[mask_tr_orch], y_tr[mask_tr_orch]

t0 = time.time()
clf_orchard = RandomForestClassifier(**dict(RF_PARAMS, class_weight='balanced')).fit(X_tr_orch, y_tr_orch)
print(f'stage-2 fit: {time.time()-t0:.1f}s  on {len(y_tr_orch):,} orchard pixels  classes={clf_orchard.classes_.tolist()}')
del X_tr_orch, y_tr_orch; gc.collect()

def cascade_predict(X_):
    pred = clf.predict(X_).astype(np.int32)
    mask = np.isin(pred, TROPICAL_ORCHARDS)
    if mask.any():
        pred[mask] = clf_orchard.predict(X_[mask]).astype(np.int32)
    return pred

## 10. Evaluation — pixel + parcel-vote

In [ ]:
from sklearn.metrics import (f1_score, accuracy_score, cohen_kappa_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt

def eval_split(name, X_, y_, predict_fn=None):
    yp = (predict_fn or clf.predict)(X_)
    print(f'\n[{name}] acc={accuracy_score(y_, yp):.4f}  '
          f'weighted_F1={f1_score(y_, yp, average="weighted", zero_division=0):.4f}  '
          f'macro_F1={f1_score(y_, yp, average="macro", zero_division=0):.4f}  '
          f'kappa={cohen_kappa_score(y_, yp):.4f}')
    print(classification_report(y_, yp, labels=CLF_CLASSES.tolist(), digits=4, zero_division=0))
    return yp

yp_va_s1 = eval_split('VAL  stage-1', X_va, y_va)
yp_te_s1 = eval_split('TEST stage-1', X_te, y_te)
yp_va    = eval_split('VAL  cascade', X_va, y_va, cascade_predict)
yp_te    = eval_split('TEST cascade', X_te, y_te, cascade_predict)

def parcel_vote(name, y_, pid_, yp_):
    df = pd.DataFrame({'pid': pid_, 'y': y_.astype(np.int32), 'yp': yp_.astype(np.int32)})
    grp = df.groupby('pid', sort=False)
    py  = grp['y'].first().to_numpy()
    ypp = grp['yp'].agg(lambda s: int(s.mode().iloc[0])).to_numpy()
    print(f'\n[{name} parcel-vote] n_parcels={len(py):,}  '
          f'acc={accuracy_score(py, ypp):.4f}  '
          f'weighted_F1={f1_score(py, ypp, average="weighted", zero_division=0):.4f}  '
          f'macro_F1={f1_score(py, ypp, average="macro", zero_division=0):.4f}')
    return py, ypp

py_va, ypp_va = parcel_vote('VAL',  y_va, pid_va, yp_va)
py_te, ypp_te = parcel_vote('TEST', y_te, pid_te, yp_te)

## 11. Confusion matrices — pixel + parcel-vote (val + test)

In [ ]:
display_labels = [f'{c} {LABEL_NAMES.get(int(c), "?")}' for c in CLF_CLASSES.tolist()]
fig, axes = plt.subplots(2, 2, figsize=(22, 18))
panels = [
    (axes[0, 0], 'VAL pixel',         y_va,  yp_va,  'Blues'),
    (axes[0, 1], 'TEST pixel',        y_te,  yp_te,  'Blues'),
    (axes[1, 0], 'VAL parcel-vote',   py_va, ypp_va, 'Greens'),
    (axes[1, 1], 'TEST parcel-vote',  py_te, ypp_te, 'Greens'),
]
for ax, title, y_true, y_pred, cmap in panels:
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred, labels=CLF_CLASSES.tolist(), display_labels=display_labels,
        normalize='true', cmap=cmap, ax=ax, colorbar=False, xticks_rotation=90, values_format='.2f')
    ax.set_title(f'{title}  (row-normalised)', fontsize=12)
plt.tight_layout()
plt.savefig(ARTIFACTS / 'cm_combined.png', dpi=120, bbox_inches='tight')
plt.show()

imp = pd.Series(clf.feature_importances_, index=ALL_FEATS).sort_values(ascending=False)
print('\nfeature importance (stage-1, top 20):')
print(imp.head(20).to_string())

## 12. Save model bundle

In [ ]:
joblib.dump({
    'clf': clf, 'clf_orchard': clf_orchard,
    'feature_cols': ALL_FEATS,
    'base_feats': FEATURE_COLS, 'delta_feats': DELTA_NAMES,
    'label_names': LABEL_NAMES, 'classes': CLF_CLASSES,
    'tropical_orchards': TROPICAL_ORCHARDS, 'rf_params': RF_PARAMS,
    'index_names': INDEX_NAMES, 'months': MONTHS, 'band_names': BAND_NAMES,
    'aoi': {'crs': REF_CRS.to_string(), 'transform': list(AOI_TF)[:6],
            'height': int(H), 'width': int(W)},
}, MODEL_PATH, compress=3)
print(f'saved -> {MODEL_PATH}  ({MODEL_PATH.stat().st_size/1e6:.1f} MB)')

## 13. Per-year prediction parquet (for Streamlit Segmentation page)

In [ ]:
df_all = pd.read_parquet(OUT_PARQ)
for idx in INDEX_NAMES:
    df_all[f'{idx} d11_10'] = (df_all[f'{idx} 11'] - df_all[f'{idx} 10']).astype(np.float32)
    df_all[f'{idx} d12_11'] = (df_all[f'{idx} 12'] - df_all[f'{idx} 11']).astype(np.float32)

for year in YEARS:
    sub = df_all[df_all['year'] == year]
    if not len(sub): continue
    X_y = sub[ALL_FEATS].to_numpy(dtype=np.float32)
    yp  = cascade_predict(X_y)
    out = pd.DataFrame({
        'row': sub['row'].astype(np.int32).to_numpy(),
        'col': sub['col'].astype(np.int32).to_numpy(),
        'label_true': sub['label'].astype(np.int32).to_numpy(),
        'label_pred': yp.astype(np.int32),
    })
    out.to_parquet(ARTIFACTS / f'preds_{year}.parquet', compression='zstd', index=False)
    agree = (out['label_true'] == out['label_pred']).mean()
    print(f'preds_{year}.parquet: {len(out):,} rows  agree={agree:.4f}  size={(ARTIFACTS / f"preds_{year}.parquet").stat().st_size/1e6:.1f} MB')
